<a href="https://colab.research.google.com/github/acoiman/pdt/blob/main/asthma_mortality/notebooks/Python/12.Asthma_Mortality_RF_Classification.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 🧩 🚧 Random Forest Classification-Hurdle-type Modelling



Hurdle-type modeling is a statistical approach designed to analyze count data, particularly when there is an excess of zeros. This model operates in two stages: the first stage determines whether the count is zero or positive using a binary classification, while the second stage models only the positive counts, often employing a regression models [1].

Following the hurdle-type modeling, in this Notebbok, we will train and evaluate a Random Forest (RF) classification model to predict the nonoccurrence (0) or occurrence (1) of asthma mortality (binary classification)  by departments in Argentina between 2001 and 2022. We will apply a walk-forward (expanding window) validation approach, which is appropriate for epidemiological studies involving time series data [2]. We will start by training the RF classification model using data from 2001 to 2006 (a 5-year window) and testing it with data from 2007. The training window will then be expanded by one year at each iteration until it spans from 2001 to 2021, with 2022 data used for testing

## 📚 Import required libraries

In [ ]:
# dataframe libraries
import pandas as pd
import numpy as np

# geospatial libraries
import geopandas as gpd
import mapclassify
from libpysal.weights import Queen
from esda.moran import Moran
from pysal.explore import esda

# plot libraries
import matplotlib.pyplot as plt
import matplotlib as mpl
from matplotlib.ticker import MaxNLocator
import seaborn as sns
from matplotlib.patches import Patch
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf

# sklearn libraries
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression, LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error,r2_score,explained_variance_score,median_absolute_error, max_error
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.ensemble import RandomForestRegressor
from sklearn.model_selection import RandomizedSearchCV
from sklearn.model_selection import GridSearchCV
from sklearn.metrics import classification_report
from sklearn.model_selection import ParameterSampler

# other libraries
import os
import re
from joblib import Parallel, delayed
import shap
from itables import init_notebook_mode, show
from lime.lime_tabular import LimeTabularExplainer
import itertools

In [ ]:
%cd work/

In [ ]:
# Set the PROJ_LIB path
os.environ['PROJ_LIB'] = "/opt/conda/envs/gds/share/proj"

In [ ]:
 # Copy-on-Write
 pd.options.mode.copy_on_write = True

##🗂 Funtions to load and transform the dataset

In [ ]:
# path to gdf
path = "pdt/asthma_mortality/data/gpkg/data.gpkg"

In [ ]:
# Funtion to create lag variables (up to 2 years)
def create_lags(df, var, max_lag=2):
  """
  Create lagged variables (up to 2 years).

    Parameters:
    df (pd.DataFrame): Original DataFrame.
    var (str): Name of the variable to create lags for.

    Returns:
    df (pd.DataFrame): DataFrame with lagged variables.
  """

  for lag in range(1, max_lag+1):
    df[f"{var}_lag{lag}"] = df[var].shift(lag)
  return df

In [ ]:
def create_ts_long_format(path):
  """
  Create long-format time series dataset from GeoDataFrame.

    Parameters:
    path (str): Path to the GeoPackage file containing the GeoDataFrame.

    Returns:
    df_ts (pd.DataFrame): Long-format time series dataset.
  """

  # Load dataset with data per department
  gdf = gpd.read_file(path)

  # Reshape df to ts long format
  years = range(2001, 2023)
  records = []
  # iterate rows
  for _, row in gdf.iterrows():
    iddpto = row["IDDPTO"]
    for year in years:
        records.append({
            "IDDPTO": iddpto,
            "YEAR": year,
            "AMR": row.get(f"AMR_{year}", np.nan),
            "PM25": row.get(f"PM25_{year}", np.nan),
            "PM10": row.get(f"PM10_{year}", np.nan),
            "NBA": row.get(f"NBA_{year}", np.nan),
            "PD": row.get(f"PD_{year}", np.nan),
            "NAGRT": row.get(f"NAGRT_{year}", np.nan),
            "NNWVT": row.get(f"NNWVT_{year}", np.nan),
            "NBUT": row.get(f"NBUT_{year}", np.nan)
            })

  # create new df from list
  df_ts = pd.DataFrame(records)

  # Create lagged variables
  for var in ["NAGRT", "NNWVT", "NBUT"]:
    df_ts = create_lags(df_ts, var)

  # Drop the initial rows with NaNs due to lagging
  df_ts = df_ts.dropna().reset_index(drop=True)

  return df_ts

## ⚙️   Training and Evaluating Random Forest Classification Model with Default Hiperparameters (Walk-forward)

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts['AMR_BIN'] = (df_ts['AMR'] > 0).astype(int)

In [ ]:
# visusalize the dataframe
init_notebook_mode(all_interactive=True)
df_ts

In [ ]:
results = []  # results list

for i in range(2006, 2022):  # start walk-forward from year 2006 to 2021
    train_years = list(range(2001, i + 1))  # expanding window
    test_year = i + 1
    print(f"Training: {train_years[0]}–{train_years[-1]}, Testing: {test_year}")

    # Split train/test by year
    train_df = df_ts[df_ts['YEAR'].isin(train_years) ]
    test_df = df_ts[df_ts['YEAR'] == test_year]

    # Define features and target
    features = ['PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT',
              'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
              'NBUT_lag1','NBUT_lag2']
    target = 'AMR_BIN'

    X_train = train_df[features]
    y_train = train_df[target]
    X_test = test_df[features]
    y_test = test_df[target]

    # Train classification model with default hiperparameters
    model = RandomForestClassifier(n_jobs=-1, random_state=42)
    model.fit(X_train, y_train)

    # Predict (labels)
    y_pred = model.predict(X_test)

    # Classification metrics
    acc = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)

    # Store results
    results.append({
        'train_years': f"{train_years[0]}-{train_years[-1]}",
        'test_year': test_year,
        'Accuracy': acc,
        'Precision': prec,
        'Recall': rec,
        'F1_Score': f1
    })

In [ ]:
# convert results into a dataframe
results_df = pd.DataFrame(results)

In [ ]:
# visualize the results
init_notebook_mode(all_interactive=True)
results_df

In [ ]:
# Calculate and display the mean and standard deviation for each evaluation metric
mean_metrics = results_df[['Accuracy', 'Precision', 'Recall', 'F1_Score']].mean()
std_metrics = results_df[['Accuracy', 'Precision', 'Recall', 'F1_Score']].std()

print("Mean of Evaluation Metrics:")
print(mean_metrics)
print("\nStandard Deviation of Evaluation Metrics:")
print(std_metrics)

Metric Value	Interpretation
* Accuracy	0.762 ± 0.017.	On average, the classifier correctly identified whether AMR was 0 or 1 about 76.2 % ± 1.7% of the time. This is decent but may be misleading if there is class imbalance (e.g., many zeros).
* Precision	0.657 ±  0.032.	On average, of all the cases where the classifier predicted AMR > 0, 65.7% ± 3.2% were correct. Upper moderate precision means some false positives (it sometimes predicts AMR as 1 when the true value is 0).
* Recall 0.573 ± 0.044.	On average, the model only identified 57.3%  ± 4.4% of true AMR > 0 cases .So it's missing nearly a third of the true positives (false negatives are moderate).
* F1 Score	0.611 ± 0.028.	The harmonic mean of precision and recall. This moderate value indicates a trade-off between missing positives and over-predicting them.

In [ ]:
# create a plot of each metric by year:
fig, ax = plt.subplots(figsize=(10, 6))
for metric in ['Accuracy', 'Precision', 'Recall', 'F1_Score']:
    ax.plot(results_df['test_year'], results_df[metric], marker='o', label=metric)

ax.set_xlabel("Test Year")
ax.set_ylabel("Score")
ax.set_title("Walk-Forward Validation Metrics (2007–2022)")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

##🛠️ Parameter Tuning Random Forest Classification Model (Walk-forward)

In this section will perform hyperparameter tunning on the RF classification model. We will use the Random Search method because Standard k-fold CV randomly shuffles observations, which could introduces data leakage in panel/time-series data [3]. We will tune the following hyperparameters [4]:

* `n_estimators` specifies the number of decision trees built in the Random Forest, where each tree is trained on a bootstrap sample of the training data and contributes a vote to the final classification. Parameter tuning is important because too few trees can produce unstable and high-variance predictions, whereas more trees generally improve robustness and generalization, although with increased computational cost.

* `max_depth` controls the maximum number of splits (depth) allowed for each decision tree, thereby limiting how complex individual trees can become. Tuning this parameter is important because deep trees can capture complex nonlinear relationships but may overfit noise in the training data, while shallow trees reduce overfitting at the risk of underfitting important patterns.

* `min_samples_split` defines the minimum number of observations required in a node before the algorithm is allowed to split that node into two child nodes. Tuning this parameter is important because small values allow aggressive splitting and highly complex trees, whereas larger values regularize the model by preventing unnecessary splits that may capture random variation rather than meaningful structure.


* `min_samples_leaf` specifies the minimum number of observations that must remain in each terminal leaf node after a split. Tuning this parameter is important because larger leaf sizes smooth decision boundaries and improve generalization by reducing sensitivity to outliers or rare cases, which is particularly useful in noisy or imbalanced datasets such as asthma mortality occurrence.

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts_pt = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts_pt['AMR_BIN'] = (df_ts_pt['AMR'] > 0).astype(int)

In [ ]:
# Predictor variables: air quality, demographic, and lagged epidemiological features
features = [
    'PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT',
    'NAGRT_lag1', 'NAGRT_lag2',
    'NNWVT_lag1', 'NNWVT_lag2',
    'NBUT_lag1',  'NBUT_lag2'
]

# Binary classification target: AMR presence/absence
target = 'AMR_BIN'

# Discrete hyperparameter grid for Random Search
# 2 × 3 × 2 × 2 = 24 total combinations per fold
param_grid = {
    'n_estimators'     : [100, 200, 300],# Number of trees in the forest
    'max_depth'        : [None, 5, 10],  # Maximum depth of each tree
    'min_samples_split': [2, 5, 10],     # Minimum samples required to split an internal node
    'min_samples_leaf' : [1, 2, 3]       # Minimum samples required at a leaf node
}

# Expand all combinations into a list of dicts
param_combinations = [
    dict(zip(param_grid.keys(), values))
    for values in itertools.product(*param_grid.values())
]

print(f"Total hyperparameter combinations per fold: {len(param_combinations)}")

In [ ]:
def evaluate_params_clf(params, X_train, y_train, X_test, y_test):
    """
    Train a Random Forest classifier with a given hyperparameter set
    and return classification evaluation metrics.
    Designed to be called in parallel via joblib.Parallel.

    Parameters
    ----------
    params   : dict  — hyperparameter combination to evaluate
    X_train  : array — training feature matrix
    y_train  : array — training target vector (binary)
    X_test   : array — test feature matrix
    y_test   : array — test target vector (binary)

    Returns
    -------
    dict with params, Accuracy, Precision, Recall, and F1-Score
    """
    # Instantiate classifier with the current hyperparameter combination
    model = RandomForestClassifier(
        n_estimators      = params['n_estimators'],
        max_depth         = params['max_depth'],
        min_samples_split = params['min_samples_split'],
        min_samples_leaf  = params['min_samples_leaf'],
        n_jobs            = -1,    # Parallelise tree fitting within each model
        random_state      = 42
    )

    # Fit on the expanding training window
    model.fit(X_train, y_train)

    # Predict binary class labels on the held-out test year
    y_pred = model.predict(X_test)

    # Compute classification metrics
    acc  = accuracy_score(y_test, y_pred)
    prec = precision_score(y_test, y_pred, zero_division=0)
    rec  = recall_score(y_test, y_pred, zero_division=0)
    f1   = f1_score(y_test, y_pred, zero_division=0)

    return {
        'params'    : params,
        'Accuracy'  : acc,
        'Precision' : prec,
        'Recall'    : rec,
        'F1_Score'  : f1
    }

In [ ]:
# Store best result per expanding window fold
results_pt = []

# Walk-forward expanding window: start training 2001–2006, test 2007
# Expands annually until training 2001–2021, test 2022
for i in range(2006, 2022):

    train_years_pt = list(range(2001, i + 1))  # Expanding training window
    test_year_pt   = i + 1

    print(f"Training: {train_years_pt[0]}–{train_years_pt[-1]}  |  Testing: {test_year_pt}")

    # Split train and test sets by year
    train_df_pt = df_ts_pt[df_ts_pt['YEAR'].isin(train_years_pt)]
    test_df_pt  = df_ts_pt[df_ts_pt['YEAR'] == test_year_pt]

    # Extract feature matrices and binary target vectors
    X_train_pt = train_df_pt[features].values
    y_train_pt = train_df_pt[target].values
    X_test_pt  = test_df_pt[features].values
    y_test_pt  = test_df_pt[target].values

    # Skip fold if either partition is empty to avoid fitting errors
    if len(X_train_pt) == 0 or len(X_test_pt) == 0:
        print(f"  Skipped: Empty train or test partition for {test_year}")
        continue

    # Evaluate all hyperparameter combinations in parallel
    # n_jobs=-1 distributes combinations across all available CPU cores
    fold_results = Parallel(n_jobs=-1, prefer='threads')(
        delayed(evaluate_params_clf)(params, X_train_pt, y_train_pt, X_test_pt, y_test_pt)
        for params in param_combinations
    )

    # Select the best combination: primary = highest Recall
    # Ties broken by highest F1-Score to balance precision trade-off
    best = max(
        fold_results,
        key=lambda x: (round(x['Recall'], 6), round(x['F1_Score'], 6))
    )

    # Store the best result for this fold
    results_pt.append({
        'train_years': f"{train_years_pt[0]}-{train_years_pt[-1]}",
        'test_year'  : test_year_pt,
        'Accuracy'   : round(best['Accuracy'],  4),
        'Precision'  : round(best['Precision'], 4),
        'Recall'     : round(best['Recall'],    4),
        'F1_Score'   : round(best['F1_Score'],  4),
        'Best_Params': best['params']
    })

In [ ]:
# Assemble results into a summary DataFrame
results_class_pt = pd.DataFrame(results_pt)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
results_class_pt

In [ ]:
# Calculate and display the mean and standard deviation for each evaluation metric
mean_metrics = results_class_pt[['Accuracy', 'Precision', 'Recall', 'F1_Score']].mean()
std_metrics = results_class_pt[['Accuracy', 'Precision', 'Recall', 'F1_Score']].std()

print("Mean of Evaluation Metrics:")
print(mean_metrics)
print("\nStandard Deviation of Evaluation Metrics:")
print(std_metrics)

Metric Value	Interpretation
* Accuracy	0.773 ± 0.026.	On average, the classifier correctly identified whether AMR was 0 or 1 about 77.3% ± 2.6% of the time. This is decent but may be misleading if there is class imbalance (e.g., many zeros).
* Precision	0.671 ±  0.048.	On average, of all the cases where the classifier predicted AMR > 0, 67.1% ± 4.8% were correct. Upper moderate precision means some false positives (it sometimes predicts AMR as 1 when the true value is 0).
* Recall 0.610 ± 0.042.	On average, the model only identified 61%  ± 4.2% of true AMR > 0 cases .  So it's missing nearly a third of the true positives (false negatives are moderate).
* F1 Score	0.638 ± 0.030.	The harmonic mean of precision and recall. This moderate value indicates a trade-off between missing positives and over-predicting them.

We observed that parameter tuning improved model performance, particularly the recall metric, which is critical in epidemiological modeling because it measures the model’s ability to correctly identify true positive cases and minimize false negatives.

In [ ]:
# create a plot of each metric by year:
fig, ax = plt.subplots(figsize=(10, 6))
for metric in ['Accuracy', 'Precision', 'Recall', 'F1_Score']:
    ax.plot(results_class_pt['test_year'], results_class_pt[metric], marker='o', label=metric)

ax.set_xlabel("Test Year")
ax.set_ylabel("Score")
ax.set_title("Walk-Forward Validation Metrics (2007–2022) with Parameter Tuning")
ax.legend()
ax.grid(True)
plt.tight_layout()
plt.show()

## 🏋️‍♂️Training Random Forest Classification Model on 2001–2021 Data and Predicting 2022 Using Tuned Hyperparameters

In this section, for the SHAP  (SHapley Additive Explanations) analysis, we will train the RF classification model on data from 2001–2021 and predict, for 2022 the nonoccurrence (0) or occurrence (1) of asthma mortality (binary classification) by departments in Argentina

In [ ]:
# apply the funtion to create long-format time series dataset from GeoDataFrame
df_ts_0122 = create_ts_long_format(path)

In [ ]:
# Binary target
df_ts_0122['AMR_BIN'] = (df_ts_0122['AMR'] > 0).astype(int)

In [ ]:
# visualize the dataframe
init_notebook_mode(all_interactive=True)
df_ts_0122

In [ ]:
# Define features and target
features = ['PM25', 'PM10', 'NBA', 'PD', 'NAGRT', 'NNWVT', 'NBUT',
            'NAGRT_lag1', 'NAGRT_lag2', 'NNWVT_lag1', 'NNWVT_lag2',
            'NBUT_lag1','NBUT_lag2']
target = 'AMR_BIN'

In [ ]:
# Training and test sets
train_df_0122 = df_ts_0122[df_ts_0122['YEAR'] <= 2021].dropna(subset=features + [target])
test_df_0122 = df_ts_0122[df_ts_0122['YEAR'] == 2022].dropna(subset=features + [target])

In [ ]:
# Define inputs
X_train_0122 = train_df_0122[features]
y_train_0122 = train_df_0122[target]

# Keep IDDPTO in test set
X_test_full_0122 = test_df_0122[['IDDPTO'] + features+ ['AMR_BIN']].copy()
y_test_0122 = test_df_0122[target].reset_index(drop=True)

In [ ]:
# # get the tuned hiperpameter of the training 2001-2021
# last_row_params = results_class_pt.iloc[-1]['Best_Params']
# formatted_params = str(last_row_params).replace("'", "").replace(":", "=")
# print(formatted_params)

In [ ]:
# Train model and fit the model
clf_0122 = RandomForestClassifier(max_depth= 10, min_samples_leaf= 3, min_samples_split= 2, n_estimators= 200, random_state=42)
clf_0122.fit(X_train_0122, y_train_0122)

In [ ]:
# Predict using only feature columns
y_pred_0122 = clf_0122.predict(X_test_full_0122[features])

#  Add prediction to test set with IDDPTO
X_test_full_0122['AMR_BIN_PRED'] = y_pred_0122

In [ ]:
# Evaluate prediction
accuracy_0122 = accuracy_score(y_test_0122, y_pred_0122)
precision_0122= precision_score(y_test_0122, y_pred_0122, zero_division=0)
recall_0122 = recall_score(y_test_0122, y_pred_0122, zero_division=0)
f1_0122 = f1_score(y_test_0122, y_pred_0122, zero_division=0)

# Print metrics
print(f"Prediction Results for 2022:")
print(f"Accuracy:  {accuracy_0122:.3f}")
print(f"Precision: {precision_0122:.3f}")
print(f"Recall:    {recall_0122:.3f}")
print(f"F1 Score:  {f1_0122:.3f}")

Metric Value	Interpretation
* Accuracy	0.775:	The classifier correctly predicted whether AMR was 0 or 1 in 2022 in about 77.5% of the time. This is decent but may be misleading if there is class imbalance (e.g., many zeros).
* Precision	0.687:	Of all the cases where the classifier predicted AMR > 0 in 2022, 68.7 % were correct. This precision means a few false positives (it sometimes predicts AMR as 1 when the true value is 0).
* Recall 0.594:	The model only identified 59.4%  of true AMR > 0 cases.  So it's missing nearly a third of the true positives (false negatives are moderate).
* F1 Score	0.637:	The harmonic mean of precision and recall. This moderate value indicates a trade-off between missing positives and over-predicting them.

In [ ]:
# Preserve only IDDPTO, CA_bin, and CA_bin_pred from X_test_full
bin_pred_result_0122 = X_test_full_0122[['IDDPTO', 'AMR_BIN', 'AMR_BIN_PRED']].copy()

In [ ]:
# Display the new DataFrame
init_notebook_mode(all_interactive=True)
show(bin_pred_result_0122)

In [ ]:
# Get predicted probabilities for both classes
y_probs_0122 = clf_0122.predict_proba(X_test_full_0122[features])

# Add a new column to bin_pred_result for the predicted probability of the predicted class
bin_pred_result_0122['AMR_BIN_PRED_PROB'] = [
    y_probs_0122[i, pred_class] for i, pred_class in enumerate(bin_pred_result_0122['AMR_BIN_PRED'])
]

In [ ]:
# Display the updated DataFrame
init_notebook_mode(all_interactive=True)
show(bin_pred_result_0122)

## 🤔 SHAP (SHapley Additive Explanations)

In this section we will interpret the contribution of each independent variable to the final predition of the RF classification model using the SHAP method.

In [ ]:
# Background dataset (commonly a sample of training data)
background_0122 = shap.sample(X_train_0122, 1000, random_state=42)

In [ ]:
# SHAP for Binary Classifier
# Use TreeExplainer with the output of the model transformed into probability space
explainer_0122 = shap.TreeExplainer(
    clf_0122,
    data=background_0122,
    model_output="probability",
    feature_perturbation="interventional"
)

In [ ]:
# get the test set
X_0122 = test_df_0122[features]

In [ ]:
# get shap values
shap_values_0122 = explainer_0122(X_0122)

In [ ]:
# get SHAP values of Class 1 (occurrence of asthma mortality)
shap_values_0122_class1 = shap_values_0122.values[:, :, 1]

In [ ]:
# Generate a SHAP summary plot to visualize the impact of features on the model's predictions
shap.summary_plot(shap_values_0122_class1, X_0122, feature_names=features,
                  show= False, cmap = "plasma", plot_size=(8, 6))
plt.title("SHAP Summary Plot RF Classification", fontsize=17)
plt.xlabel("SHAP Values (probability)", size=12)
plt.ylabel("Features", size=12)
plt.ylabel("Features")
plt.tight_layout()
# save figure
#plt.savefig('pdt/asthma_mortality/data/images/figures/shap_rf_class.png', dpi=300)
plt.show();

### Extract SHAP Values

In [ ]:
# SHAP values as dataframe
shap_df_0122 = pd.DataFrame(
    shap_values_0122_class1,
    columns=features,
    index=X_0122.index
)

In [ ]:
# rename all columns of shap_df like _sahp
shap_df_0122.columns = [f"{col}_SHAP" for col in shap_df_0122.columns]

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
shap_df_0122.head()

In [ ]:
# X dataframe
X_df_0122 = X_0122.copy()

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
X_df_0122.head()

In [ ]:
# join shap_df_0122 and X_df_0122 by index
shap_df_join_0122 = shap_df_0122.join(X_df_0122)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
shap_df_join_0122.head()

In [ ]:
# from shap_df_join_0122 select all records where PM10_SHAP is between 0.00 and 0.1 (moderate-high positive PM10-SHAP values)
pm10_pos_0122 =  shap_df_join_0122[(shap_df_join_0122['PM10_SHAP'] >= 0.0) &  (shap_df_join_0122['PM10_SHAP'] <= 0.1)][["PM10_SHAP", "PM10"]]

In [ ]:
# get statistics
init_notebook_mode(all_interactive=True)
pm10_pos_0122.describe()

In [ ]:
# from shap_df_join_0122 select all records where PD_SHAP is between 0.35 and 0.45 (high positive PD-SHAP values)
pd_high_pos_0122 =  shap_df_join_0122[(shap_df_join_0122['PD_SHAP'] >= 0.35) &  (shap_df_join_0122['PD_SHAP'] <= 0.45)][["PD_SHAP", "PD"]]

In [ ]:
# get statistics
init_notebook_mode(all_interactive=True)
pd_high_pos_0122.describe()

In [ ]:
# get mean Baseline prediction (0, 1)
clf_0122.predict_proba(X_train_0122).mean(axis=0)

 Among the high-impact features, high population density (PD) and moderate-to-high $PM_{10}$ concentrations  were the most consistent predictors of a higher probability of asthma mortality occurrence.

* PD (Population Density). Although the relationship at low-to-moderate PD  values does not appear linear, PD is the strongest predictor displayed, with high positive SHAP contributions across its range. At high population density levels (mean ≈ 5218.29), the model predicts an asthma mortality occurrence probability of 0.37, compared with the baseline probability of ≈ 0.34. The model therefore suggests that dense living environments are associated with higher predicted asthma mortality occurrence.

* $PM_{10}$  shows a clear, positive  association with a low-moderate predicted probability of asthma mortality occurrence. Across the range represented by the SHAP values, this relationship appears approximately linear. Moderate-to-high $PM_{10}$  levels (mean ≈ 128.71) contribute positively  to the model prediction, increasing the predicted probability by approximately 4 percentage points relative to the baseline probability (≈ 0.34).

## 🗣️ LIME (Local Interpretable Model-agnostic Explanations)

LIME (Local Interpretable Model-agnostic Explanations) is a machine learning technique that explains individual predictions from complex "black box" models (like Random Forest) by approximating them locally with simpler, interpretable models (like linear models) [6].

In [ ]:
feature_names = X_train_0122.columns.tolist()
class_names = ['Non-occurrence', 'Ocurrence']

The code block bellow will allow us to initialize the LIME Tabular Explainer Key design choices:

* mode='classification'

* discretize_continuous=True (recommended for environmental predictors)

* training_data=X_train. LIME must be fitted on the training data distribution, not on the test set alone.

In [ ]:
# Initialize LIME
lime_explainer = LimeTabularExplainer(
    training_data=X_train_0122.values,
    feature_names=feature_names,
    class_names=class_names,
    mode='classification',
    discretize_continuous=True,
    verbose=True,
    random_state=42,
)

In [ ]:
# Select test data (2022)
test_df = df_ts_0122[df_ts_0122['YEAR'] == 2022]

# Department identifiers
iddpto_series = test_df_0122['IDDPTO'].copy()

# Feature matrix
X_test_lime = test_df_0122.drop(
    columns=['IDDPTO', 'YEAR', 'AMR', 'AMR_BIN']
)

In [ ]:
# Random Forest prediction wrapper
def rf_predict_proba_named(x):
    return clf_0122.predict_proba(
        pd.DataFrame(x, columns=X_test_lime.columns)
    )

In [ ]:
# LIME explanations

lime_contributions = []

for i in range(len(X_test_lime)):

    # Observation
    instance = X_test_lime.iloc[i].values
    iddpto = iddpto_series.iloc[i]

    # RF prediction
    pred_proba = rf_predict_proba_named(instance.reshape(1, -1))[0]

    # Predicted class (0 or 1)
    bin_pred = np.argmax(pred_proba)

    # Probability of predicted class
    bin_pred_prob = pred_proba[bin_pred]

    # LIME explanation
    exp = lime_explainer.explain_instance(
        data_row=instance,
        predict_fn=rf_predict_proba_named,
        num_features=len(X_test_lime.columns),
        top_labels=2
    )

    # Extract contributions for predicted class
    for feature, contribution in exp.as_list(label=bin_pred):

        lime_contributions.append({
            'IDDPTO': iddpto,
            'feature': feature,
            'lime_contribution': contribution,
            'bin_pred': bin_pred,
            'bin_pred_prob': bin_pred_prob
        })

In [ ]:
# Convert to dataframe
lime_by_dept = pd.DataFrame(lime_contributions)

In [ ]:
# Visualize dataframe
init_notebook_mode(all_interactive=True)
show(lime_by_dept)

In [ ]:
# select only those departments where bin_pred == 1 (occurence of asthma mortality)
lime_contributions_df = lime_by_dept[lime_by_dept['bin_pred'] == 1]

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
lime_contributions_df

In our LIME  analysis we will only take into account contributions to the predicted probability of the positive class, defined as: asthma mortality occurrence (class = 1). In epidemiology, we are often  concerned with the patterns of disease occurrence in human populations [5].

LIME contributions should be interpreted as follows:

* Positive LIME contribution
→ Increases the predicted probability of asthma mortality occurrence

* Negative LIME contribution
→ Decreases the predicted probability of asthma mortality occurrence

Department `06028` was selected as an illustrative case for interpreting local LIME feature contributions. The rule `PD > 25.23` had the largest positive LIME contribution (≈ +0.37), indicating that, for this department, population density was the most influential predictor shifting the model toward the asthma mortality occurrence class. The Random Forest predicted an asthma mortality occurrence probability of approximately 0.88. Relative to the other predictors considered by the model, PD therefore played the largest role in this local prediction. This describes the model explanation for department `06028`; it does not establish that population density causally drives asthma mortality.

It is important to note that the feature intervals shown by LIME are not thresholds learned by the Random Forest model. Instead, LIME creates these intervals by discretizing continuous variables according to their distribution in the training data. Therefore, the intervals should be interpreted as local explanatory ranges used by LIME to explain a specific prediction, rather than as decision rules learned by the model [6].



In [ ]:
# Aggregate LIME contributions by department and predictor
lime_by_dept_agg = (
    lime_contributions_df
    .groupby(['IDDPTO', 'feature'], as_index=False)
    .agg(
        mean_lime_contribution=('lime_contribution', 'mean'),# Mean contribution per predictor per department
        abs_mean_contribution=('lime_contribution', lambda x: np.mean(np.abs(x)))
    )
)


In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
lime_by_dept_agg

In [ ]:
# Pivot to wide format (mapping )
lime_pivot = lime_by_dept_agg.pivot(
    index='IDDPTO',
    columns='feature',
    values='mean_lime_contribution'
).reset_index()


In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
lime_pivot.head()

### Mapping LIME contributions

In [ ]:
# Load dataset with data per department
gdf = gpd.read_file("pdt/asthma_mortality/data/gpkg/data.gpkg")

In [ ]:
# Merge LIME pivot with geodataframe
gdf_lime = gdf.merge(
    lime_pivot,
    on='IDDPTO',
    how='left'
)

In [ ]:
# visualize dataframe
init_notebook_mode(all_interactive=True)
gdf_lime

In [ ]:
# gdf_lime length
len(gdf_lime)

#### Mapping LIME contributions for Population Density

In [ ]:
# get columns names with Population Density (PD) LIME features
pd_long_columns = [col for col in gdf_lime.columns if 'PD' in col and len(col) > 7]
print(pd_long_columns)

In [ ]:
# copy pd_long_columns
pd_long_columns_titles = pd_long_columns.copy()

# add "intermediate-density departments:" to pd_long_columns[0], and  "high-density departments:" to pd_long_columns[1]
pd_long_columns_titles[0] = "Intermediate-density departments\n" + pd_long_columns[0] + " people/km²"
pd_long_columns_titles[1] = "High-density departments\n" + pd_long_columns[1] + " people/km²"

In [ ]:
print(pd_long_columns_titles)

In [ ]:
# define columns to map
columns = pd_long_columns

# define subplot titles
titles = pd_long_columns_titles

# Calculate global min and max across all relevant columns for the common color bar
global_min = gdf_lime[columns].min().min()
global_max = gdf_lime[columns].max().max()

# Create a figure and a grid of subplots
fig, axes = plt.subplots(1, len(pd_long_columns), figsize=(15, 10))
axes = axes.flatten()

# Create a ScalarMappable for the common colorbar
# Ensure the colormap is created correctly
current_cmap = mpl.colormaps.get_cmap("viridis")
sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=global_min, vmax=global_max), cmap=current_cmap)
sm._A = [] # Required for matplotlib 3.1 and later

for ax, col, title in zip(axes, columns, titles):
    gdf_valid = gdf_lime[gdf_lime[col].notna()]
    gdf_nan = gdf_lime[gdf_lime[col].isna()]

    # Plot departments with NaN values (outline only)
    gdf_nan.plot(
        ax=ax,
        facecolor="none",
        edgecolor="black",
        linewidth=0.5
    )

    # Plot departments with valid LIME values
    gdf_valid.plot(
        ax=ax,
        column=col,
        cmap="viridis",
        linewidth=0.5,
        edgecolor="black",
        legend=False, # No individual legend for subplots
        vmin=global_min, # Apply global min/max for consistent coloring
        vmax=global_max  # Apply global min/max for consistent coloring
    )

    ax.set_title(title, fontsize=10)
    ax.set_axis_off()

# Add a single colorbar for the entire figure on the right
cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7]) # [left, bottom, width, height] - Changed width to 0.01 for thinner bar
cbar = fig.colorbar(
    sm,
    cax=cbar_ax,
    orientation="vertical"
)
# Setup colorbar
cbar.outline.set_linewidth(0.5)
cbar.ax.tick_params(labelsize=8)
cbar.locator = MaxNLocator(nbins=4)
cbar.update_ticks()
cbar.set_label("LIME contributions", fontsize=8, rotation=270, labelpad=15)

# Add the overall title
#plt.suptitle('LIME contributions of Population Density (PD) for Predicted Asthma Mortality Occurence in 2022-Argentina', fontsize=14, y=0.98)

# Create 'No Data' patch for the legend
no_data_patch = Patch(facecolor='none', edgecolor='black', label='No Data')

# Add the 'No Data' legend in the lower center of the figure
fig.legend(handles=[no_data_patch], loc='lower center', bbox_to_anchor=(0.5, 0.1), fontsize=10.5)

# save figure
#plt.savefig('pdt/asthma_mortality/data/images/figures/lime_pd_rf_class.png', dpi=300)

plt.show()

The maps show spatial heterogeneity in the local contribution of population density (PD) to the Random Forest model’s predictions of asthma mortality occurrence in Argentina in 2022. Departments with intermediate population density (≈ 7.62 < PD ≤ 25.23) have LIME contributions near zero or only weakly positive, suggesting limited predictive influence within this range. In contrast, high population density (≈ PD > 25.23) is associated with strong positive LIME contributions, shifting model predictions toward asthma mortality occurrence. These departments are concentrated primarily in northern and central-eastern Argentina, including several urbanized areas. Overall, the results suggest a non-linear model relationship: PD becomes an important positive predictor beyond this threshold.

#### Mapping LIME contributions for $PM_{10}$

In [ ]:
# get columns names with PM10 LIME features
pm10_long_columns = [col for col in gdf_lime.columns if 'PM10' in col and len(col) > 9]
print(pm10_long_columns)

In [ ]:
def extract_pm10_range(s):
    """
    Extracts the lower and upper bounds from a PM10 range string.
    Returns a tuple (priority, lower_bound, upper_bound) for custom sorting.
    """
    numbers = [float(n) for n in re.findall(r'\d+\.?\d*', s)]

    if 'PM10 <=' in s:
        # Priority 0 for 'PM10 <=' ranges, then (lower_bound=-inf, upper_bound=number)
        return (0, float('-inf'), numbers[0])
    elif '< PM10 <=' in s:
        # Priority 1 for 'X < PM10 <=' ranges, then (lower_bound=number1, upper_bound=number2)
        return (1, numbers[0], numbers[1])
    elif 'PM10 >' in s:
        # Priority 2 for 'PM10 >' ranges, then (lower_bound=number, upper_bound=+inf)
        return (2, numbers[0], float('inf'))
    else:
        # Fallback for unexpected formats with lowest priority
        return (3, float('-inf'), float('inf'))

In [ ]:
# Sort the list programmatically using the custom key
pm10_long_columns_sorted = sorted(pm10_long_columns, key=extract_pm10_range)

In [ ]:
print(pm10_long_columns_sorted)

In [ ]:
# copy pm10_long_columns
pm10_long_columns_titles = pm10_long_columns_sorted.copy()

# add category names
pm10_long_columns_titles[0] = "Low\n" + pm10_long_columns_titles[0] + " μg/m³"
pm10_long_columns_titles[1] = "Lower-Intermediate\n" + pm10_long_columns_titles[1] + " μg/m³"
pm10_long_columns_titles[2] = "Upper-Intermediate\n" + pm10_long_columns_titles[2] + " μg/m³"
pm10_long_columns_titles[3] = "High\n" + pm10_long_columns_titles[3] + " μg/m³"

In [ ]:
# define columns to map
columns = pm10_long_columns_sorted

# define subplot titles
titles = pm10_long_columns_titles

# Calculate global min and max across all relevant columns for the common color bar
global_min = gdf_lime[columns].min().min()
global_max = gdf_lime[columns].max().max()

# Create a figure and a grid of subplots
fig, axes = plt.subplots(1, len(pm10_long_columns_sorted), figsize=(15, 10))
axes = axes.flatten()

# Create a ScalarMappable for the common colorbar
# Ensure the colormap is created correctly
current_cmap = mpl.colormaps.get_cmap("viridis")
sm = mpl.cm.ScalarMappable(norm=mpl.colors.Normalize(vmin=global_min, vmax=global_max), cmap=current_cmap)
sm._A = [] # Required for matplotlib 3.1 and later

for ax, col, title in zip(axes, columns, titles):
    gdf_valid = gdf_lime[gdf_lime[col].notna()]
    gdf_nan = gdf_lime[gdf_lime[col].isna()]

    # Plot departments with NaN values (outline only)
    gdf_nan.plot(
        ax=ax,
        facecolor="none",
        edgecolor="black",
        linewidth=0.5
    )

    # Plot departments with valid LIME values
    gdf_valid.plot(
        ax=ax,
        column=col,
        cmap="viridis",
        linewidth=0.5,
        edgecolor="black",
        legend=False, # No individual legend for subplots
        vmin=global_min, # Apply global min/max for consistent coloring
        vmax=global_max  # Apply global min/max for consistent coloring
    )

    ax.set_title(title, fontsize=10)
    ax.set_axis_off()

# Add a single colorbar for the entire figure on the right
cbar_ax = fig.add_axes([0.92, 0.15, 0.01, 0.7]) # [left, bottom, width, height] - Changed width to 0.01 for thinner bar
cbar = fig.colorbar(
    sm,
    cax=cbar_ax,
    orientation="vertical"
)

# Setup colorbar
cbar.outline.set_linewidth(0.5)
cbar.ax.tick_params(labelsize=8)
cbar.locator = MaxNLocator(nbins=4)
cbar.update_ticks()
cbar.set_label("LIME contributions", fontsize=8, rotation=270, labelpad=15)

# Add the overall title
#plt.suptitle('LIME contributions of $PM_{10}$ for Predicted Asthma Mortality Occurence in 2022 Argentina', fontsize=14, y=0.85)

# Create 'No Data' patch for the legend
no_data_patch = Patch(facecolor='none', edgecolor='black', label='No Data')

# Add the 'No Data' legend in the lower center of the figure
fig.legend(handles=[no_data_patch], loc='lower center', bbox_to_anchor=(0.5, 0.2), fontsize=10)

# save figure
#plt.savefig('pdt/asthma_mortality/data/images/figures/lime_PM10_rf_class.png', dpi=300)

plt.show()

The four maps group municipalities by $PM_{10}$ concentration interval and color them by the local LIME contribution of $PM_{10}$ to predicted asthma mortality occurrence (Class 1). The color bar shows a signed effect: purple tones are negative contributions, near-zero values are blue or teal, and yellow tones are positive contributions. The interval maps represent exposure strata, while the colors represent $PM_{10}$'s local explanatory direction and magnitude within each stratum.

For the two lower intervals ($PM_{10}$ ≤ 114.33 and 114.33 < $PM_{10}$ ≤ 117.40), highlighted municipalities are sparse and fragmented, with most contributions in the negative (purple) range. These negative values appear in small clusters in the northern and central-western portions of the country, and in isolated eastern locations. This suggests that, at these $PM_{10}$ levels, $PM_{10}$ generally contributes a weak to moderate negative effect on Class 1 predictions, with limited geographic continuity.

In the intermediate interval (**117.40 < PM10 ≤ 122.57**), the mapped municipalities become more numerous and more spatially connected, especially across the central-eastern belt. Contributions shift toward near-zero to mildly negative (blue/teal), indicating attenuation of the earlier negative effect. This interval forms the clearest regional aggregation, consistent with a transition zone where PM10’s local effect is less strongly directional.

In the intermediate interval (117.40 < $PM_{10}$  ≤ 122.57), mapped municipalities are more numerous and spatially connected, especially across the central-eastern belt. Contributions shift toward near-zero to mildly negative (blue/teal), attenuating the earlier negative effect. The interval forms a contiguous regional aggregation, consistent with a transition zone where $PM_{10}$ 's local effect is less directional.

In the highest interval ( $PM_{10}$ > 122.57), a compact east-central/eastern cluster shows the most positive contributions (yellow), forming a hotspot where  $PM_{10}$ is a strong positive contributor to Class 1 predictions. The maps show a gradient from predominantly negative contributions at lower  $PM_{10}$ intervals to positive contributions at the highest interval, with regional clustering strongest in the intermediate and high strata.

## 📚 References

[1] Hasanah, S., S. Abdullah, and Dian Lestari. "Bayesian method for hurdle regression." ICSA-Int. Conf. Stat. Anal. Vol. 2019. 2019.

[2] Utku, A., & Akcayol, M. A. (2024). Spread patterns of COVID-19 in European countries: hybrid deep learning model for prediction and transmission analysis. Neural Computing and Applications, 36(17), 10201-10217.


[3] S. Albelali and M. Ahmed, “Hidden Leaks in Time Series Forecasting: How Data Leakage Affects LSTM Evaluation Across Configurations and Validation Strategies,” Dec. 07, 2025, arXiv: arXiv:2512.06932. doi: 10.48550/arXiv.2512.06932.

[4] scikit-learn developers, “RandomForestClassifier,” scikit-learn. Accessed: Jun. 25, 2026. [Online]. Available: https://scikit-learn/stable/modules/generated/sklearn.ensemble.RandomForestClassifier.html

[5] D. E. Lilienfeld et al., Eds., “Laying The Foundations: The Epidemiologic Approach To Disease,” in Foundations Of Epidemiology, Oxford University Press, 1994, p. 0. doi: 10.1093/oso/9780195050356.003.0001.

[6]. Molnar, C. (2020). Interpretable machine learning. Lulu. com.






